# 16 — ¿Qué contexto recupera C2 en las preguntas de hecho puntual (E1)?

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`13-jss-anchura-de-citas.ipynb`](13-jss-anchura-de-citas.ipynb) ·
[`07-jss-runner-experimento-b.ipynb`](07-jss-runner-experimento-b.ipynb) ·
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md) · issue #146

---

> **Estado: análisis descriptivo sobre datos de desarrollo, como insumo para la decisión de #146.**
> Se usan las 21 preguntas de calibración de ES2015 (14 de ellas de tipo E1) y los registros de
> tres corridas de humo ya versionadas: C1 (`humo-c1-es2015`), C2 (`humo-c2-es2015`) y C2 con la
> corrección del adaptador que ubica las citas (`humo-c2-es2015-conceptos`). ES2015 es una **serie
> de desarrollo** (ADR 0005) y la muestra es pequeña: **ninguna cifra de este notebook es un
> resultado de la tesis**. No se corrió ningún modelo ni experimento; solo se leen las trazas
> guardadas. El notebook **no decide** si C2 debe recuperar fuentes en bruto: ordena la evidencia
> para que el equipo decida.

# 1. La pregunta

**En palabras llanas.** En el Experimento B se le hacen las mismas preguntas a dos sistemas. C1
busca fragmentos de la transcripción de la reunión y responde con ellos. C2 responde desde una
«base compilada» que OpenKOS construye antes: decisiones, resúmenes de temas y otros objetos
extraídos de la transcripción. Las preguntas E1 piden un hecho puntual (por ejemplo, «¿qué precio
de venta se fijó?»): la respuesta está escrita en un solo lugar de la reunión.

**Por qué importa.** La hipótesis H1 espera una diferencia **pequeña o nula** entre C2 y C1 en E1,
porque en un hecho puntual la transcripción ya lo dice y compilar poco puede agregar. La
propuesta agrega: si C2 pierde en E1, hay una **regresión que explicar**. En las corridas de humo
C1 acertó 10 de 14 preguntas E1 y C2 acertó 5 de 14, es decir, C2 pierde. Este notebook busca
explicar la pérdida mirando **qué texto recibió el modelo de C2** al responder, y a qué etapa
(OE4) se atribuye cada error.

**Tres preguntas concretas.**

1. ¿De qué tipo es el contexto que C2 recupera en E1: transcripción en bruto, decisiones o
   resúmenes de tema?
2. ¿Se relaciona ese tipo de contexto con acertar o fallar, y con lo que hizo C1?
3. ¿Cómo se clasifican los errores de C2 en E1 según la atribución de error de la tesis (OE4:
   extracción, recuperación o síntesis)?

## Glosario mínimo

| Término | Significado aquí |
|---|---|
| **OpenKOS** | Sistema que construye la base compilada a partir de transcripciones y responde consultas sobre ella. |
| **Base compilada** (*bundle*) | Conjunto de objetos que OpenKOS guarda: `decisions/`, `concepts/`, `events/`, `people/`, `projects/` y `sources/`. |
| **`sources/`** | La transcripción **en bruto**, importada tal cual. No es un resultado de la extracción. |
| **`decisions/`** | Una decisión extraída por el modelo (una frase y un extracto). |
| **`concepts/`** | Resumen de un tema, redactado por el modelo. |
| **Bloque de contexto** | Cada objeto de la base que la consulta entrega al modelo para que responda. Una consulta entrega 5. |
| **E1** | Pregunta de hecho puntual (la respuesta está en un solo lugar). |
| **OE4** | Objetivo que atribuye cada respuesta incorrecta de C2 a una etapa: **extracción** (el hecho no quedó bien en la base), **recuperación** (estaba y no se recuperó) o **síntesis** (se recuperó y la respuesta lo contradice). |
| **Veredicto del juez** | Si la respuesta coincide con la de referencia, según el modelo juez (`gpt-oss:20b`). |

## Mapa de pasos

1. Cargar las preguntas y las tres corridas de humo (sección 2).
2. Ver **un caso real completo**: pregunta, contexto recuperado y respuesta (sección 3).
3. Medir la **composición del contexto** por tipo de bloque (sección 4).
4. Cruzar esa composición con el **acierto** de C2 y de C1 (sección 5).
5. Reunir la **evidencia para clasificar los errores** bajo OE4 (sección 6).
6. Hallazgos y preguntas abiertas para el equipo (secciones 7 y 8).

Todas las cifras salen de `afg.experiments.c2_context` (con pruebas en `tests/test_c2_context.py`).

# 2. Datos

In [1]:
from dataclasses import asdict

import pandas as pd

from afg.evaluation.judge_calibration_worksheet import WORKSHEET_PATH
from afg.experiments.c2_context import (
    KINDS,
    context_rows,
    correctness_by_presence,
    error_evidence,
    same_retrieval,
    source_excerpts,
    stratum_composition,
)
from afg.experiments.questions import load_experiment_questions
from afg.experiments.runner import RECORDS_FILE, load_records
from afg.shared.paths import EXPERIMENT_B_DIR

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)

RUNS = {
    "C1": EXPERIMENT_B_DIR / "humo-c1-es2015",
    "C2": EXPERIMENT_B_DIR / "humo-c2-es2015",
    "C2 + conceptos": EXPERIMENT_B_DIR / "humo-c2-es2015-conceptos",
}
for needed in (WORKSHEET_PATH, *(run / RECORDS_FILE for run in RUNS.values())):
    if not needed.exists():
        raise SystemExit(
            f"No se encontro {needed}.\n"
            "Se generan con `uv run afg experiment b --help` (corridas de humo, requieren Ollama "
            "y, para C2, OpenKOS) y con la hoja de calibracion de ES2015."
        )

QUESTIONS = {q.id: q for q in load_experiment_questions(WORKSHEET_PATH)}
RECORDS = {name: load_records(run / RECORDS_FILE, latest_only=True) for name, run in RUNS.items()}
for name, records in RECORDS.items():
    print(f"{name}: {len(records)} registros")
print(
    "Las dos corridas de C2 recuperaron los mismos bloques, en el mismo orden:",
    same_retrieval(RECORDS["C2"], RECORDS["C2 + conceptos"]),
)

C1: 21 registros
C2: 21 registros
C2 + conceptos: 21 registros
Las dos corridas de C2 recuperaron los mismos bloques, en el mismo orden: True


**Lectura.** Las dos corridas de C2 recuperan **exactamente los mismos bloques** para las 21
preguntas (`True`). Se diferencian solo en la corrección del adaptador que **ubica las citas** en la
transcripción (commit `0db4d02`), que afecta la métrica de citación (notebook 13) y no el contexto
ni los aciertos. Por eso el resto del notebook usa la corrida `humo-c2-es2015-conceptos` y la
comparación entre las dos configuraciones queda resuelta: no difieren en lo que aquí se mide.

# 3. Un caso real: «¿Qué precio de venta se fijó para el control remoto?»

La pregunta `q002` es un E1 típico. La respuesta de referencia es «Twenty-five Euros» y el hecho
está en la reunión ES2015a. La tabla muestra los cinco bloques que C2 entregó al modelo y la
respuesta que dio.

In [2]:
EXAMPLE = next(r for r in RECORDS["C2 + conceptos"] if r["question_id"] == "ES2015.q002")
print("Pregunta:", QUESTIONS["ES2015.q002"].text)
print("Referencia:", QUESTIONS["ES2015.q002"].reference_answer)
print("Respuesta de C2:", EXAMPLE["answer"]["text"])
print("Veredicto del juez (correcta):", EXAMPLE["verdict"]["correct"])
print(
    "Respuesta de C1 correcta:",
    next(r for r in RECORDS["C1"] if r["question_id"] == "ES2015.q002")["verdict"]["correct"],
)
pd.DataFrame(
    [
        {
            "orden": i,
            "bloque": b["concept_id"],
            "caracteres": len(b["text"]),
            "recortado": b["excerpted"],
            "comienzo del texto": b["text"].split("\n", 3)[-1][:80].replace("\n", " "),
        }
        for i, b in enumerate(EXAMPLE["trace"]["context_blocks"], start=1)
    ]
)

Pregunta: What selling price was set for the remote control?
Referencia: Twenty-five Euros.
Respuesta de C2: Matching concepts were found in the compiled bundle, but none of them answers this question.
Veredicto del juez (correcta): False
Respuesta de C1 correcta: True


,orden,bloque,caracteres,recortado,comienzo del texto
0,1,concepts/remote-control-features-and-user-experience,1727,True,"A discussion on the features and user experience of remote controls, including a"
1,2,sources/es2015b,1743,True,Raw source imported from 'ES2015b.md' as raw/ES2015b.md; full text embedded verb
2,3,concepts/design-of-a-remote-control,2012,True,"A discussion on the design considerations for a remote control, focusing on user"
3,4,decisions/the-remote-control-will-be-marketed-as-easier-to-find-compared-to-other-remo...,712,False,remote control visibility: The remote control will be marketed as easier to find
4,5,sources/es2015a,1886,True,Raw source imported from 'ES2015a.md' as raw/ES2015a.md; full text embedded verb


**Lectura.** C2 recibió dos resúmenes de tema, una decisión sobre cómo se comercializaría el
control y **dos transcripciones en bruto recortadas**; ninguno de los cinco bloques menciona el
precio, y el modelo, con razón, declaró que no podía responder. C1, en cambio, buscó fragmentos
de la reunión y acertó. El caso muestra la forma del problema: el hecho existe en la reunión, pero
la consulta no lo trajo a la vista del modelo.

# 4. Composición del contexto por tipo de bloque

Cada fila agrega las respuestas de un estrato. «Con fuentes» cuenta las respuestas cuyo contexto
incluye al menos un bloque `sources/` (transcripción en bruto). Las medias son bloques de cada
tipo por respuesta (suman 5 en total, con `other` = eventos, personas y proyectos).

In [3]:
ROWS = context_rows(RECORDS["C2 + conceptos"], RECORDS["C1"], QUESTIONS)
composition = pd.DataFrame([asdict(s) for s in stratum_composition(ROWS)]).round(2)
composition.rename(
    columns={
        "stratum": "estrato",
        "answers": "respuestas",
        "correct": "C2 acierta",
        "c1_correct": "C1 acierta",
        "with_sources": "con fuentes",
        "with_decisions": "con decisiones",
        "mean_sources": "sources (media)",
        "mean_decisions": "decisions (media)",
        "mean_concepts": "concepts (media)",
        "mean_other": "other (media)",
    }
)

,estrato,respuestas,C2 acierta,C1 acierta,con fuentes,con decisiones,sources (media),decisions (media),concepts (media),other (media)
0,E1,14,5,10,12,12,1.36,1.71,1.50,0.43
1,E3,1,0,0,1,1,1.00,1.00,2.00,1.00
2,E2,2,1,2,2,1,2.00,0.50,2.00,0.50
3,E4,4,4,4,3,3,0.75,1.25,2.75,0.25


In [4]:
EXCERPTS = source_excerpts(RECORDS["C2 + conceptos"])
print(
    f"Bloques sources/ en las 21 consultas: {EXCERPTS.blocks}; recortados: {EXCERPTS.excerpted}; "
    f"largo medio: {EXCERPTS.mean_chars:.0f} caracteres"
)

Bloques sources/ en las 21 consultas: 27; recortados: 27; largo medio: 1910 caracteres


**Lectura.** En E1 el contexto de C2 **no es sobre todo transcripción en bruto**: de los 5 bloques,
1,36 son `sources/`, 1,71 son `decisions/`, 1,50 son `concepts/` y 0,43 son otros. 12 de las 14
respuestas E1 incluyen al menos una transcripción y 12 incluyen al menos una decisión. La
descripción inicial del issue («sobre todo fuentes en bruto y conceptos») vale para la traza que
se revisó entonces, pero no resume las 14: el contexto es una mezcla de los tres tipos.
Un dato sobre las transcripciones: los 27 bloques `sources/` están **recortados** (≈1.900
caracteres, el comienzo de una reunión entera). Que la fuente en bruto llegue al modelo no
significa que llegue la parte de la reunión que contiene el hecho.

# 5. ¿Se relaciona el tipo de contexto con acertar?

In [5]:
E1 = [r for r in ROWS if r.stratum == "E1"]
split = pd.DataFrame([asdict(correctness_by_presence(E1, kind)) for kind in KINDS])
split.rename(
    columns={
        "kind": "tipo de bloque",
        "with_kind": "respuestas con el tipo",
        "correct_with_kind": "de ellas, correctas",
        "without_kind": "respuestas sin el tipo",
        "correct_without_kind": "de ellas, correctas (sin el tipo)",
    }
)

,tipo de bloque,respuestas con el tipo,"de ellas, correctas",respuestas sin el tipo,"de ellas, correctas (sin el tipo)"
0,sources,12,4,2,1
1,decisions,12,3,2,2
2,concepts,14,5,0,0
3,other,5,3,9,2


In [6]:
pd.DataFrame(
    [
        {
            "pregunta": r.question,
            "C2 correcta": r.correct,
            "C1 correcta": r.c1_correct,
            "resultado de la consulta": r.outcome,
            "bloques (orden)": " > ".join(r.kinds),
            "fuente de la reunión de referencia": r.reference_source_retrieved,
        }
        for r in E1
    ]
)

,pregunta,C2 correcta,C1 correcta,resultado de la consulta,bloques (orden),fuente de la reunión de referencia
0,q002,False,True,no_answer_in_context,concepts > sources > concepts > decisions > sources,True
1,q003,False,True,answered,decisions > other > concepts > sources > decisions,False
2,q004,False,True,no_answer_in_context,sources > decisions > decisions > concepts > decisions,False
3,q006,False,True,no_answer_in_context,decisions > sources > sources > concepts > decisions,True
4,q007,True,True,answered,decisions > decisions > sources > sources > concepts,True
5,q008,True,False,answered,decisions > concepts > concepts > decisions > concepts,False
6,q010,True,True,answered,concepts > concepts > sources > sources > other,True
7,q011,False,False,answered,decisions > sources > concepts > decisions > sources,True
8,q013,True,True,answered,decisions > other > sources > other > concepts,True
9,q016,True,False,answered,concepts > concepts > sources > other > concepts,True


**Lectura.** **El tipo de bloque no separa los aciertos de los fallos con esta muestra.** Con al
menos una transcripción en bruto acierta 4 de 12 respuestas; sin ninguna, 1 de 2. Con al menos una
decisión, 3 de 12; sin ninguna, 2 de 2 (dos casos, demasiado poco para leer una tendencia). Todas
las respuestas llevan algún resumen de tema, así que ese tipo no se puede comparar. Lo que sí
muestra la tabla por pregunta es la **pérdida frente a C1**: C2 falla 7 de las 10 preguntas E1 que
C1 acierta (`q002`, `q003`, `q004`, `q006`, `q017`, `q020`, `q023`), mientras que C2 acierta 2
(`q008`, `q016`) que C1 no, y ambas fallan en `q011` y `q026`. De las nueve respuestas E1 que C2
falla, cinco incluyen la transcripción de la reunión que contiene el hecho de referencia, y aun así
no responden bien.

# 6. Evidencia para la atribución de error (OE4)

La regla de OE4 es una cascada de tres preguntas: ¿el hecho está bien en la base?; si está, ¿se
recuperó?; si se recuperó, ¿el modelo lo contradijo? (`afg.evaluation.attribution`). Desde una
traza guardada solo se puede comprobar **una parte**:

- Si el texto de **referencia completo** aparece en algún bloque recuperado, el hecho estaba en la
  base y se recuperó, así que la regla da **síntesis**. Se comprueba con una coincidencia léxica de
  términos (aproximada: «25» y «twenty-five» no coinciden), por lo que sirve como pista y no como
  prueba.
- Si **no** aparece, la traza no distingue entre **extracción** (el hecho nunca quedó bien en la
  base) y **recuperación** (estaba en la base y no se trajo): para saberlo hay que mirar la base
  compilada de OpenKOS, que no se versiona. Esa etiqueta **la pone una persona** (CONTRIBUTING §7.6);
  la columna `etapa por regla` queda vacía y no se la asigna ningún modelo.

La tabla reúne, para cada respuesta E1 incorrecta de C2, lo que hace falta para etiquetarla.

In [7]:
EVIDENCE = error_evidence(E1, RECORDS["C2 + conceptos"], QUESTIONS)
table = pd.DataFrame(
    [
        {
            "pregunta": e.question,
            "estrato": e.stratum,
            "resultado": e.outcome,
            "bloques (orden)": " > ".join(e.kinds),
            "cobertura de la referencia": round(e.reference_coverage, 2),
            "referencia completa en": ", ".join(e.visible_in) or "ningún bloque",
            "fuente de la reunión": e.reference_source_retrieved,
            "etapa por regla": e.rule_stage.value if e.rule_stage else None,
        }
        for e in EVIDENCE
    ]
)
table

,pregunta,estrato,resultado,bloques (orden),cobertura de la referencia,referencia completa en,fuente de la reunión,etapa por regla
0,q002,E1,no_answer_in_context,concepts > sources > concepts > decisions > sources,0.67,ningún bloque,True,NaN
1,q003,E1,answered,decisions > other > concepts > sources > decisions,1.00,concepts,False,synthesis
2,q004,E1,no_answer_in_context,sources > decisions > decisions > concepts > decisions,0.25,ningún bloque,False,NaN
3,q006,E1,no_answer_in_context,decisions > sources > sources > concepts > decisions,0.29,ningún bloque,True,NaN
4,q011,E1,answered,decisions > sources > concepts > decisions > sources,0.53,ningún bloque,True,NaN
5,q017,E1,answered,decisions > sources > sources > concepts > decisions,0.50,ningún bloque,True,NaN
6,q020,E1,no_answer_in_context,decisions > concepts > decisions > decisions > decisions,0.60,ningún bloque,False,NaN
7,q023,E1,answered,decisions > concepts > sources > sources > other,0.25,ningún bloque,True,NaN
8,q026,E1,answered,concepts > decisions > decisions > sources > concepts,0.60,ningún bloque,False,NaN


In [8]:
n = len(EVIDENCE)
print(f"Respuestas E1 incorrectas de C2: {n}")
print(
    "con la referencia completa visible (síntesis por regla):",
    sum(e.rule_stage is not None for e in EVIDENCE),
)
print(
    "sin la referencia completa visible (etiqueta pendiente de una persona):",
    sum(e.rule_stage is None for e in EVIDENCE),
)
print("de ellas, abstenciones:", sum(e.abstained and e.rule_stage is None for e in EVIDENCE))
print(
    "con la transcripción de la reunión de referencia en el contexto:",
    sum(bool(e.reference_source_retrieved) for e in EVIDENCE),
)

Respuestas E1 incorrectas de C2: 9
con la referencia completa visible (síntesis por regla): 1
sin la referencia completa visible (etiqueta pendiente de una persona): 8
de ellas, abstenciones: 4
con la transcripción de la reunión de referencia en el contexto: 5


**Lectura.** **La regla objetiva atribuye una sola de las nueve fallas y deja ocho para una
persona.** La única con la referencia completa visible es `q003`: el mercado «internacional» está en
el resumen de tema, pero el modelo contestó «jóvenes adultos» a partir de otra decisión del
contexto (un competidor plausible, no un olvido); la regla la clasifica como síntesis. Las otras
ocho tienen una cobertura de entre 0,25 y 0,67 de los términos de la referencia, cuatro de ellas
con una abstención («no hay respuesta en el contexto»). En estas ocho **la traza no permite
separar extracción de recuperación**; la lectura más económica (a confirmar por una persona) es
que el contexto trajo temas y decisiones cercanos pero no la frase que contiene el hecho, y que
las transcripciones en bruto recortadas tampoco la incluyen. Hay además un efecto del léxico:
«Twelve Euros and fifty cents» o «Kinetic energy» no se reconocen si el texto escribe «12,50» o
usa otra palabra, lo que subestima la cobertura.

# 7. Hallazgos

1. **La pérdida de C2 en E1 está en la muestra:** 5 de 14 frente a 10 de 14 de C1. Con 14
   preguntas de una sola serie de desarrollo es una dirección, no una estimación.
2. **El contexto de C2 en E1 es una mezcla**, no sobre todo transcripción en bruto: por consulta, en
   promedio 1,4 bloques de transcripción, 1,7 de decisiones, 1,5 de resúmenes de tema y 0,4 de otros
   tipos. Las dos configuraciones de C2 recuperan lo mismo.
3. **Las transcripciones que llegan están recortadas** (≈1.900 caracteres): son el comienzo de la
   reunión, no necesariamente el pasaje del hecho.
4. **El tipo de bloque no explica por sí solo el acierto** con esta muestra (con y sin
   `sources/`: 4 de 12 y 1 de 2).
5. **Bajo OE4, de nueve fallas E1, una es síntesis por regla y ocho necesitan etiqueta humana.**
   No se puede decir todavía si el grueso es extracción o recuperación sin abrir la base compilada.

# 8. Preguntas abiertas para el equipo

- **La decisión de fondo (de jss):** ¿las fuentes en bruto deben poder recuperarse en C2 al
  responder, o C2 debe responder solo desde la base compilada? Este notebook aporta que las
  transcripciones ya se recuperan (recortadas) y que no bastan por sí solas; no la resuelve. Debe
  coordinarse con [openkos#1560](https://github.com/jasonssdev/openkos/issues/1560), que mide los
  modelos de respuesta.
- ¿Quién etiqueta las ocho fallas E1 pendientes y con qué acceso a la base compilada
  (`data/interim/openkos/…`, no versionada)? Sin ese paso la cifra principal de OE4 no se puede
  calcular.
- Si C2 solo puede responder desde la base compilada, ¿una comparación justa con C1 en E1 debería
  informarse aparte, ya que C2 queda privado de la transcripción por diseño?
- ¿El límite de 5 bloques por consulta es parte de la configuración de C2 que se registra o un
  parámetro a explorar? Con 5 bloques de ≈2.000 caracteres el contexto se llena con temas
  cercanos.

*Límites:* una sola serie de desarrollo; 14 preguntas E1; un modelo generativo (`qwen3:8b`); una
corrida por pregunta; el juez es un modelo y su veredicto no es una etiqueta humana.